# 01 -- Baseline trade audit

Paired script: `analysis/analyse_baseline.py`. Per the reproducibility contract's rule 1
("every notebook has a paired `.py` pipeline containing the actual logic"), all the real
logic lives in that script and its shared modules (`metrics.py`, `trade_math.py`) -- this
notebook only builds a fixture and calls into it.

**Uses clearly-labelled SYNTHETIC fixture data.** Neither baseline EA (V6.37/V8.11) has a
committed real trade export -- `01_BASELINE/` contains only source code, screenshots, and
set files (see `TASK-028_PYTHON_STATISTICAL_LAB.md`'s Risks section). Per reproducibility
rule 7, the real-data run is marked PENDING at the end of this notebook rather than
fabricated.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.analyse_baseline import run

In [ ]:
# Synthetic fixture: 4 trades, hand-computable win rate / expectancy / profit
# factor / max drawdown (same numbers verified by hand in
# tests/test_analyse_baseline.py::test_summary_hand_computed).
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_baseline_demo_"))
trades_csv = tmp_dir / "trades.csv"

pd.DataFrame(
    [
        {"trade_id": "t1", "symbol": "XAUUSD", "is_long": "True",
         "entry_time": "2026-07-21T00:00:00Z", "exit_time": "2026-07-21T01:00:00Z",
         "entry_price": 100.0, "exit_price": 102.0, "stop_price": 98.0, "profit": 40.0},
        {"trade_id": "t2", "symbol": "XAUUSD", "is_long": "True",
         "entry_time": "2026-07-21T02:00:00Z", "exit_time": "2026-07-21T03:00:00Z",
         "entry_price": 100.0, "exit_price": 99.0, "stop_price": 98.0, "profit": -20.0},
        {"trade_id": "t3", "symbol": "XAUUSD", "is_long": "False",
         "entry_time": "2026-07-21T04:00:00Z", "exit_time": "2026-07-21T05:00:00Z",
         "entry_price": 100.0, "exit_price": 95.0, "stop_price": 102.0, "profit": 50.0},
        {"trade_id": "t4", "symbol": "XAUUSD", "is_long": "True",
         "entry_time": "2026-07-21T06:00:00Z", "exit_time": "2026-07-21T07:00:00Z",
         "entry_price": 100.0, "exit_price": 98.0, "stop_price": 98.0, "profit": -20.0},
    ]
).to_csv(trades_csv, index=False)
print(f"Synthetic trades written to: {trades_csv}")

In [ ]:
summary = run(trades_csv, starting_balance=1000.0, output_json=tmp_dir / "summary.json",
              repo_path=PROJECT_ROOT.parents[1])

print(f"n_trades                  = {summary['n_trades']}")
print(f"win_rate                  = {summary['win_rate']['value']:.4f} (95% CI [{summary['win_rate']['ci_lower']:.4f}, {summary['win_rate']['ci_upper']:.4f}])")
print(f"expectancy ($)            = {summary['expectancy_dollars']['value']:.2f}")
print(f"expectancy (R)            = {summary['expectancy_r']['value']:.4f}")
print(f"profit_factor             = {summary['profit_factor']:.4f}")
print(f"max_balance_drawdown_pct  = {summary['max_balance_drawdown_pct']:.4f}")
print(f"final_balance             = {summary['final_balance']:.2f}")

assert summary["n_trades"] == 4
assert abs(summary["win_rate"]["value"] - 0.5) < 1e-9
assert abs(summary["expectancy_dollars"]["value"] - 12.5) < 1e-9
assert abs(summary["profit_factor"] - 2.25) < 1e-9
# balance_curve = [1000, 1040, 1020, 1070, 1050] -- largest abs/pct decline
# both at peak=1040 -> trough=1020: 20, ~1.923% (hand-traced in
# tests/test_analyse_baseline.py::test_summary_hand_computed).
assert abs(summary["max_balance_drawdown_abs"] - 20.0) < 1e-9
assert abs(summary["max_balance_drawdown_pct"] - (20.0 / 1040.0)) < 1e-9
assert abs(summary["final_balance"] - 1050.0) < 1e-9

## Real-data run: PENDING

No real baseline trade export exists yet. Once one is produced (via a future task bridging
a real MT5 statement export into `analyse_baseline.py`'s documented CSV schema), re-run this
notebook's second cell pointed at that file instead of the synthetic fixture.